# Local reconstruction confirmation

Run this notebook on **Colab T4**. It compares local task specialists and one shared masked pretraining per training seed. Estimated T4 time: **2-2.5 hours**, not yet measured for this complete run. Training plus selection has a cumulative three-hour ceiling.

Prepare `calolab-reco-confirmation.zip`. Drive stores verified checkpoints. Run the cells in order and return the final results ZIP. All checkpoints and predictions stay outside the code repository. **No reserved-test events are included.**


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

SMOKE = os.environ.get("CALOLAB_CONFIRMATION_SMOKE") == "1"
BUNDLE_SHA256 = "0e758bf20eda5cde456ff77ca218a396705437bff24cff63c718484e24509fe9"
ROOT = Path(os.environ.get("CALOLAB_CONFIRMATION_ROOT", "/content/calolab-confirmation"))
ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = "cpu" if SMOKE else "cuda"
print("Mode:", "tiny CPU smoke, not scientific" if SMOKE else "full T4 confirmation")

## Persistence and input

Drive access is requested by Colab. Keep the same input ZIP when resuming. A changed bundle uses a separate directory. A recovery copies only verified committed checkpoints; prediction files and plots can be rebuilt.


In [ ]:
if SMOKE:
    DRIVE = ROOT / "drive-smoke"
else:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/calolab-reco/confirmation")
DRIVE = DRIVE / BUNDLE_SHA256[:16]
DRIVE.mkdir(parents=True, exist_ok=True)
bundle = Path(
    os.environ.get("CALOLAB_CONFIRMATION_BUNDLE", str(ROOT / "calolab-reco-confirmation.zip"))
)
if not bundle.exists() and (DRIVE / "input.zip").exists():
    shutil.copyfile(DRIVE / "input.zip", bundle)
if not bundle.exists():
    from google.colab import files

    uploaded = files.upload()
    if "calolab-reco-confirmation.zip" not in uploaded:
        raise RuntimeError("Upload calolab-reco-confirmation.zip, then rerun this cell.")
    bundle.write_bytes(uploaded["calolab-reco-confirmation.zip"])
    del uploaded
if hashlib.sha256(bundle.read_bytes()).hexdigest() != BUNDLE_SHA256:
    raise RuntimeError("Input checksum differs. Use the matching notebook and bundle.")
if not (DRIVE / "input.zip").exists():
    staged = DRIVE / "input.partial"
    shutil.copyfile(bundle, staged)
    if hashlib.sha256(staged.read_bytes()).hexdigest() != BUNDLE_SHA256:
        raise RuntimeError("Drive input copy differs.")
    staged.replace(DRIVE / "input.zip")
print("Verified input and Drive backup:", BUNDLE_SHA256)

In [ ]:
# Bootstrap only after verifying the complete input ZIP against the fixed checksum.
workspace = ROOT / "workspaces" / BUNDLE_SHA256
workspace.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle) as archive:
    manifest = json.loads(archive.read("MANIFEST.json"))
    if set(archive.namelist()) != set(manifest["files"]) | {"MANIFEST.json"}:
        raise RuntimeError("Unexpected input members.")
    for name, expected in manifest["files"].items():
        target = (workspace / name).resolve()
        if not target.is_relative_to(workspace.resolve()):
            raise RuntimeError("Unsafe archive path.")
        value = archive.read(name)
        if hashlib.sha256(value).hexdigest() != expected:
            raise RuntimeError("Input member checksum differs.")
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() and target.read_bytes() != value:
            raise RuntimeError("Workspace differs; use a fresh workspace.")
        target.write_bytes(value)
(workspace / "INPUT_MANIFEST.json").write_text(json.dumps(manifest))
if SMOKE:
    python = Path(os.environ["CALOLAB_CONFIRMATION_PYTHON"])
else:
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.12.6"], check=True)
    subprocess.run(
        ["uv", "sync", "--locked", "--extra", "cuda", "--no-dev", "--python", "3.12"],
        cwd=workspace,
        check=True,
    )
    python = workspace / ".venv/bin/python"
env = dict(
    os.environ,
    PYTHONPATH=str(workspace / "src"),
    PYTHONUNBUFFERED="1",
    MPLCONFIGDIR=str(ROOT / "matplotlib"),
    MPLBACKEND="Agg",
)


def run_module(module, *args):
    subprocess.run(
        [str(python), "-u", "-m", "calolab_reco.confirmation." + module, *map(str, args)],
        cwd=workspace,
        env=env,
        check=True,
    )


run_dir = ROOT / "runs" / (BUNDLE_SHA256[:16] + ("-smoke" if SMOKE else ""))
backup = DRIVE / ("run-smoke" if SMOKE else "run")
if not run_dir.exists() and backup.exists():
    shutil.copytree(backup, run_dir)
print("Workspace:", workspace)
print("Run:", run_dir)
print("Backup:", backup)

## Protocol check

The main condition uses three seeds. Each seed has two CNN specialists, two direct Transformer specialists, **one** shared pretraining and two separate fine-tunings. Clean and noise-only controls have one seed and no additional pretraining.

**Reading check:** why does a position offset from the observed maximum not leak the true impact? Because the origin is calculated from measured crystals, without labels. Why keep clean controls? They show whether an advantage depends on the measurement scenario.


In [ ]:
protocol = json.loads((workspace / "configs/confirmation.json").read_text())
print(
    json.dumps(
        {
            k: protocol[k]
            for k in (
                "training_seeds",
                "primary_regime",
                "control_regimes",
                "updates",
                "pretraining_updates",
                "energy_loss",
                "position_loss",
                "reserved_test",
            )
        },
        indent=2,
    )
)
subprocess.run(
    [
        str(python),
        "-c",
        "from calolab_reco.pilot import configure; import json; print(json.dumps(configure('"
        + DEVICE
        + "', 20260925), indent=2))",
    ],
    env=env,
    cwd=workspace,
    check=True,
)

## Verify recovery before full training

Tiny subsets are used only to compare uninterrupted updates with interrupted updates resumed in a new process. We compare weights, optimizer, schedule, sampler, random states and selected checkpoint. A failure stops execution before the full study.


In [ ]:
replay = ROOT / "recovery-checks" / (BUNDLE_SHA256[:16] + "-" + DEVICE)
replay = replay / str(time.time_ns())
run_module("verification", "--workspace", workspace, "--output", replay, "--device", DEVICE)
replay_result = json.loads((replay / "verification.json").read_text())
assert all(value["exact"] for value in replay_result.values())
print("Fresh-process replay passed for pretraining and fine-tuning.")

## Train and compare

Progress is printed every 50 updates and validation every 220 updates. Each committed checkpoint is copied to Drive and verified. Rerunning resumes completed work.

**Reading check:** pretraining loss compares hidden measurements with their reconstructions; supervised losses compare predictions with incident labels. The two objectives answer different questions. The temporary decoder is removed before either fine-tuning.


In [ ]:
env["MPLBACKEND"] = "Agg"  # Render files without inheriting the notebook display backend.
if not all(value["exact"] for value in replay_result.values()):
    raise RuntimeError("Recovery checks must pass first.")
arguments = ["--workspace", workspace, "--output", run_dir, "--device", DEVICE, "--backup", backup]
if SMOKE:
    arguments.append("--smoke")
run_module("training", *arguments)
summary = json.loads((run_dir / "summary.json").read_text())
print("Complete:", summary["complete"], "| Recorded phases:", len(summary["cases"]))
print("A ceiling stop is a partial run. Export it for review; do not silently increase the budget.")
(run_dir / "recovery_verification.json").write_text(json.dumps(replay_result, indent=2))

## Read the results

Lower errors are better. Dashed lines are matched classical references. Points are individual training seeds. The fine-tuning curve excludes pretraining; the report also includes that cost and counts shared pretraining once for the two tasks.

**Reading check:** does a faster fine-tuning mean less total computation? Only if its saving also covers pretraining. A strong position result does not imply an energy gain.


In [ ]:
from IPython.display import Image, Markdown, display

run_module("reporting", "--run", run_dir)
report_text = (run_dir / "RESULTS.md").read_text()
for caption, name in [("Task comparisons", "comparison.png"), ("Learning curves", "learning.png")]:
    report_text = report_text.replace(f"![{caption}](figures/{name})", "")
display(Markdown(report_text))
for filename in ("comparison.png", "learning.png"):
    display(Image(filename=str(run_dir / "figures" / filename)))
reconstruction = sorted((run_dir / "figures").glob("*pretraining.png"))
if reconstruction:
    display(Image(filename=str(reconstruction[0])))

## Verify, save and return the ZIP

Export works for complete or partial runs. The verifier checks input identity, completion, selected checkpoints and validation predictions against the original IDs and targets. It never evaluates the reserved test.


In [ ]:
stamp = time.time_ns()
result_zip = ROOT / (
    "calolab-reco-confirmation-" + ("smoke" if SMOKE else "results") + f"-{stamp}.zip"
)
run_module("transport", "export", "--run", run_dir, "--output", result_zip)
run_module("transport", "verify", "--bundle", bundle, "--result", result_zip)
backup_zip = DRIVE / result_zip.name
shutil.copyfile(result_zip, backup_zip)
assert (
    hashlib.sha256(result_zip.read_bytes()).hexdigest()
    == hashlib.sha256(backup_zip.read_bytes()).hexdigest()
)
print("Verified export:", result_zip.name)
print("SHA256:", hashlib.sha256(result_zip.read_bytes()).hexdigest())
if not SMOKE:
    from google.colab import files

    files.download(str(result_zip))